# Latency Attribution — Experiment Explorer

Interactive notebook for exploring per-experiment latency data, comparing distributions,
and generating quick visualizations. This complements the batch analysis scripts in
`analysis/scripts/`.

**Blueprint Reference:** §04 Measurement Plan, §06 Correlation Analysis

In [ ]:
import json, csv, os, glob
from pathlib import Path
from datetime import datetime

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams

rcParams['figure.figsize'] = (14, 6)
rcParams['font.size'] = 11
%matplotlib inline

DATA_DIR = Path('../data')
print(f'Data directory: {DATA_DIR.resolve()}')
print(f'Experiments found: {sorted([d.name for d in DATA_DIR.iterdir() if d.is_dir()])}')

## 1. Load Experiment Data

Helper functions to load ghz JSON output and eBPF CSV data.

In [ ]:
def load_ghz_run(exp_name, run_idx=0):
    """Load a single ghz JSON run."""
    exp_dir = DATA_DIR / exp_name
    jsons = sorted(exp_dir.glob('*.json'))
    if run_idx >= len(jsons):
        print(f'Run {run_idx} not found in {exp_name}')
        return None
    with open(jsons[run_idx]) as f:
        return json.load(f)

def extract_latencies(ghz_data, unit='ms'):
    """Extract per-request latencies from ghz details array."""
    details = ghz_data.get('details', [])
    lats = [d['latency'] for d in details if d.get('status') == 'OK' and d.get('latency', 0) > 0]
    lats = np.array(lats, dtype=float)
    if unit == 'ms':  lats /= 1e6
    elif unit == 'us': lats /= 1e3
    return lats

def load_aggregated():
    """Load the aggregated results CSV."""
    agg_path = DATA_DIR / 'aggregated_results.csv'
    if not agg_path.exists():
        print('aggregated_results.csv not found')
        return None
    rows = []
    with open(agg_path) as f:
        for r in csv.DictReader(f):
            rows.append(r)
    return rows

def load_ebpf():
    """Load eBPF per-experiment data."""
    path = DATA_DIR / 'ebpf_per_experiment.csv'
    if not path.exists():
        return {}
    ebpf = {}
    with open(path) as f:
        for r in csv.DictReader(f):
            ebpf[r['experiment']] = {k: float(v) for k, v in r.items() if k != 'experiment'}
    return ebpf

agg = load_aggregated()
ebpf = load_ebpf()
print(f'Loaded {len(agg) if agg else 0} aggregated rows, {len(ebpf)} eBPF profiles')

## 2. Quick CDF Comparison

Compare latency CDFs across selected experiments.

In [ ]:
# ── Select experiments to compare ──
COMPARE = ['e1-baseline', 'e3a-cfs-tight', 'e7-full-stress', 'e13-cpu-pinning', 'e15-full-isolation']

fig, ax = plt.subplots(figsize=(12, 6))
for exp in COMPARE:
    data = load_ghz_run(exp)
    if data is None: continue
    lats = extract_latencies(data)
    sorted_lats = np.sort(lats)
    cdf = np.arange(1, len(sorted_lats) + 1) / len(sorted_lats)
    ax.plot(sorted_lats, cdf, label=f'{exp} (p99={np.percentile(lats,99):.1f}ms, n={len(lats)})')

ax.set_xlabel('Latency (ms)')
ax.set_ylabel('CDF')
ax.set_title('Latency CDF Comparison')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_xlim(left=0)
plt.tight_layout()
plt.show()

## 3. Percentile Summary Table

Tabular view of key percentiles (p50, p95, p99, p99.9) for all experiments.

In [ ]:
ALL_EXPS = sorted([d.name for d in DATA_DIR.iterdir() if d.is_dir() and d.name.startswith('e')])

print(f'{"Experiment":<30} {"N":>8} {"p50":>8} {"p95":>8} {"p99":>8} {"p99.9":>8} {"max":>8}')
print('─' * 85)
for exp in ALL_EXPS:
    data = load_ghz_run(exp)
    if data is None: continue
    lats = extract_latencies(data)
    if len(lats) == 0: continue
    print(f'{exp:<30} {len(lats):>8} {np.percentile(lats,50):>8.1f} '
          f'{np.percentile(lats,95):>8.1f} {np.percentile(lats,99):>8.1f} '
          f'{np.percentile(lats,99.9):>8.1f} {np.max(lats):>8.1f}')

## 4. Per-Experiment Deep Dive

Select a single experiment to inspect time-series, histogram, and run consistency.

In [ ]:
# ── Change this to explore different experiments ──
TARGET_EXP = 'e7-full-stress'

data = load_ghz_run(TARGET_EXP)
lats = extract_latencies(data)
details = data.get('details', [])

# Extract timestamps
timestamps = []
for d in details:
    if d.get('status') == 'OK':
        try:
            dt = datetime.fromisoformat(d['timestamp'].replace('Z', '+00:00')[:32])
            timestamps.append(dt.timestamp())
        except:
            timestamps.append(0)
ts = np.array(timestamps)
ts = ts - ts.min()  # relative

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Time-series
axes[0].scatter(ts, lats, s=0.5, alpha=0.3, color='steelblue')
axes[0].set_xlabel('Time (s)')
axes[0].set_ylabel('Latency (ms)')
axes[0].set_title(f'{TARGET_EXP} — Time Series')
axes[0].grid(True, alpha=0.3)

# Histogram
axes[1].hist(lats, bins=100, color='steelblue', alpha=0.7, edgecolor='white')
axes[1].axvline(np.percentile(lats, 99), color='red', linestyle=':', label=f'p99={np.percentile(lats,99):.1f}ms')
axes[1].set_xlabel('Latency (ms)')
axes[1].set_ylabel('Count')
axes[1].set_title('Histogram')
axes[1].legend()

# Box plot across runs
exp_dir = DATA_DIR / TARGET_EXP
run_lats = []
for jf in sorted(exp_dir.glob('*.json')):
    with open(jf) as f:
        rd = json.load(f)
    rl = extract_latencies(rd)
    run_lats.append(rl)
axes[2].boxplot(run_lats, showfliers=False)
axes[2].set_xlabel('Run #')
axes[2].set_ylabel('Latency (ms)')
axes[2].set_title(f'Run Consistency ({len(run_lats)} runs)')

fig.suptitle(f'{TARGET_EXP} — Deep Dive (n={len(lats)})', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. eBPF Signal Correlation

Scatter plots of kernel signals vs app-level p99 across experiments.

In [ ]:
if ebpf:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    
    exps_with_both = [e for e in ebpf if e in [d.name for d in DATA_DIR.iterdir() if d.is_dir()]]
    p99s = []
    for exp in exps_with_both:
        d = load_ghz_run(exp)
        if d:
            l = extract_latencies(d)
            p99s.append(np.percentile(l, 99) if len(l) > 0 else 0)
        else:
            p99s.append(0)
    
    signals = [('rqdelay_p99_us', 'Wakeup Delay p99 (µs)'),
               ('softirq_count', 'Softirq Count'),
               ('tcp_retransmit', 'TCP Retransmits')]
    
    for ax, (key, label) in zip(axes, signals):
        vals = [ebpf[e].get(key, 0) for e in exps_with_both]
        ax.scatter(vals, p99s, s=60, edgecolors='black', zorder=3)
        for i, exp in enumerate(exps_with_both):
            ax.annotate(exp.split('-')[0].upper(), (vals[i], p99s[i]),
                       fontsize=7, xytext=(3, 3), textcoords='offset points')
        ax.set_xlabel(label)
        ax.set_ylabel('App p99 (ms)')
        ax.grid(True, alpha=0.3)
    
    fig.suptitle('eBPF Kernel Signal vs Application p99', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('No eBPF data loaded')